# LLNet — Stacked Sparse Denoising Autoencoder for Low-light Enhancement / 저조도 향상을 위한 적층 희소 디노이징 오토인코더

**Paper**: Lore, K. G., Akintayo, A., Sarkar, S., "LLNet: A Deep Autoencoder Approach to Natural Low-Light Image Enhancement", *Pattern Recognition* 61, 650-662 (2017). DOI: 10.1016/j.patcog.2016.06.008

이 노트북은 **소형 LLNet** 을 처음부터 구현한다. 학습은 200 iteration 이내, 영상은 128×128, patch는 17×17로 제한해 CPU에서 5분 이내에 실행된다.

1. cameraman 합성 데이터: gamma 어둡힘 + Gaussian noise
2. 17×17 patch 데이터로더
3. SSDA (3-layer encoder/decoder + KL sparsity)
4. Adam 최적화 with KL sparsity loss
5. 전체 영상에 patch-wise 적용 → 결과 시각화 + PSNR/SSIM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from skimage import data
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from skimage.transform import resize
from typing import Tuple

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = torch.device('cpu')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## Part 1: Synthetic low-light dataset / 합성 저조도 데이터

- Source: scikit-image `cameraman` (256×256), resize to 128×128.
- Corruption: $I_{\text{train}} = n(g(I))$ with $g(I) = I^{\gamma}$, $\gamma \sim \mathrm{Unif}(2, 5)$, and additive Gaussian noise $\sigma \in [0, 0.1]$.

In [ ]:
def darken_and_noise(image: np.ndarray,
                     gamma: float = 3.0,
                     sigma: float = 0.05) -> np.ndarray:
    """Apply gamma darkening and Gaussian noise to a [0,1] image.

    Args:
        image: 2-D image, values in [0, 1].
        gamma: Gamma exponent (>1 darkens).
        sigma: Std of additive Gaussian noise.

    Returns:
        Corrupted image clipped to [0, 1].
    """
    darkened = np.power(np.clip(image, 0, 1), gamma)
    noisy = darkened + sigma * rng.standard_normal(darkened.shape)
    return np.clip(noisy, 0, 1).astype(np.float32)


img = data.camera().astype(np.float32) / 255.0
img = resize(img, (128, 128), anti_aliasing=True).astype(np.float32)
low = darken_and_noise(img, gamma=3.0, sigma=0.05)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Reference (clean)  PSNR={psnr_fn(img, img):.1f} dB')
axes[0].axis('off')
axes[1].imshow(low, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Low-light (gamma=3, sigma=0.05)  PSNR={psnr_fn(img, low):.1f} dB')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Part 2: 17×17 patch dataset with random gamma + noise / 패치 데이터셋

각 patch마다 새로운 random gamma와 noise를 적용하여 LLNet 논문의 합성 절차를 재현.

In [ ]:
PATCH = 17
STRIDE = 4
N_PATCHES = 800  # small for CPU runtime


def extract_patches(image: np.ndarray, patch: int, stride: int) -> np.ndarray:
    """Extract overlapping patches from a 2-D image.

    Args:
        image: Source image.
        patch: Patch side length.
        stride: Stride between patches.

    Returns:
        Array of shape (n_patches, patch, patch).
    """
    h, w = image.shape
    out = []
    for i in range(0, h - patch + 1, stride):
        for j in range(0, w - patch + 1, stride):
            out.append(image[i:i+patch, j:j+patch])
    return np.stack(out, axis=0)


clean_patches = extract_patches(img, PATCH, STRIDE)
# Repeat to get more samples, each with fresh random corruption
reps = max(1, N_PATCHES // len(clean_patches))
clean_rep = np.tile(clean_patches, (reps, 1, 1))[:N_PATCHES]

# Apply random gamma + noise per patch
gammas = rng.uniform(2.0, 5.0, size=len(clean_rep))
sigmas = rng.uniform(0.0, 0.08, size=len(clean_rep))
dark_rep = np.empty_like(clean_rep)
for k in range(len(clean_rep)):
    dark_rep[k] = darken_and_noise(clean_rep[k], gammas[k], sigmas[k])

X = torch.from_numpy(dark_rep.reshape(len(dark_rep), -1)).float()
Y = torch.from_numpy(clean_rep.reshape(len(clean_rep), -1)).float()
ds = TensorDataset(X, Y)
dl = DataLoader(ds, batch_size=64, shuffle=True)
print(f'patches: {len(X)}, dim: {X.shape[1]}')

## Part 3: Stacked Sparse Denoising Autoencoder / 적층 희소 디노이징 오토인코더

LLNet 본문에 가까운 구조 (단순화):
- input 289 → hidden 256 → 128 → bottleneck 64 → 128 → 256 → output 289
- Sigmoid activations
- KL sparsity term on bottleneck during training

In [ ]:
class LLNet(nn.Module):
    """Small SSDA for low-light enhancement.

    Args:
        d_in: Flattened input dimension.
        hidden: Sequence of hidden sizes from outer encoder to bottleneck.
    """

    def __init__(self, d_in: int = 289, hidden: Tuple[int, ...] = (256, 128, 64)):
        super().__init__()
        sizes = (d_in,) + tuple(hidden)
        enc = []
        for a, b in zip(sizes[:-1], sizes[1:]):
            enc.append(nn.Linear(a, b))
            enc.append(nn.Sigmoid())
        self.encoder = nn.Sequential(*enc)
        dec = []
        for a, b in zip(reversed(sizes[1:]), reversed(sizes[:-1])):
            dec.append(nn.Linear(a, b))
            dec.append(nn.Sigmoid())
        self.decoder = nn.Sequential(*dec)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x)
        return self.decoder(h)

    def bottleneck(self, x: torch.Tensor) -> torch.Tensor:
        """Return the bottleneck activations for sparsity regularisation."""
        return self.encoder(x)


model = LLNet(d_in=PATCH * PATCH).to(device)
print(model)
print(f'Trainable params: {sum(p.numel() for p in model.parameters())}')

## Part 4: Training loop with KL sparsity / 학습

Loss: MSE reconstruction + $\beta \cdot \mathrm{KL}(\rho \| \hat\rho)$ + L2 weight decay.

$\rho = 0.05$ (target sparse activation), $\beta = 0.01$, ≤200 iterations.

In [ ]:
def kl_sparsity(activations: torch.Tensor, rho: float = 0.05) -> torch.Tensor:
    """KL divergence between the target sparsity rho and the empirical mean activation.

    Args:
        activations: Bottleneck activations (batch, hidden).
        rho: Target average activation in (0, 1).

    Returns:
        Scalar KL penalty.
    """
    rho_hat = activations.mean(dim=0).clamp(1e-6, 1 - 1e-6)
    rho_t = torch.tensor(rho, device=activations.device)
    return torch.sum(rho_t * torch.log(rho_t / rho_hat) +
                     (1 - rho_t) * torch.log((1 - rho_t) / (1 - rho_hat)))


opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-5)
MAX_ITER = 200
history = []
iters = 0
model.train()
while iters < MAX_ITER:
    for xb, yb in dl:
        if iters >= MAX_ITER:
            break
        xb = xb.to(device)
        yb = yb.to(device)
        recon = model(xb)
        h = model.bottleneck(xb)
        loss_recon = ((recon - yb) ** 2).mean()
        loss_kl = kl_sparsity(h, rho=0.05)
        loss = loss_recon + 0.01 * loss_kl
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append((float(loss_recon), float(loss_kl)))
        iters += 1

history = np.array(history)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history[:, 0], label='reconstruction MSE')
ax.plot(history[:, 1], label='KL sparsity', alpha=0.6)
ax.set_xlabel('iteration')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.3)
ax.set_title('LLNet training losses')
plt.tight_layout()
plt.show()

## Part 5: Patch-wise inference on the full image / 전체 영상 추론

전체 저조도 영상을 17×17 patch (stride 3)로 분할 → LLNet 통과 → 평균하여 재조립.

In [ ]:
def patchwise_infer(model: nn.Module, image: np.ndarray, patch: int, stride: int) -> np.ndarray:
    """Run the LLNet on overlapping patches and reassemble.

    Args:
        model: Trained LLNet.
        image: 2-D input image in [0, 1].
        patch: Patch side length.
        stride: Stride between patches.

    Returns:
        Enhanced 2-D image, same shape as input.
    """
    model.eval()
    h, w = image.shape
    accum = np.zeros_like(image, dtype=np.float64)
    counts = np.zeros_like(image, dtype=np.float64)
    patches, positions = [], []
    for i in range(0, h - patch + 1, stride):
        for j in range(0, w - patch + 1, stride):
            patches.append(image[i:i+patch, j:j+patch].ravel())
            positions.append((i, j))
    X = torch.from_numpy(np.stack(patches)).float()
    with torch.no_grad():
        Y = model(X.to(device)).cpu().numpy()
    for (i, j), p in zip(positions, Y):
        accum[i:i+patch, j:j+patch] += p.reshape(patch, patch)
        counts[i:i+patch, j:j+patch] += 1.0
    counts[counts == 0] = 1
    return np.clip(accum / counts, 0, 1).astype(np.float32)


low_test = darken_and_noise(img, gamma=3.0, sigma=0.05)
enhanced = patchwise_infer(model, low_test, PATCH, stride=3)

# Baselines: histogram equalisation, gamma adjustment
from skimage import exposure
he = exposure.equalize_hist(low_test)
ga = np.power(low_test, 1/3.0)

fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, im, title in zip(axes,
                          [img, low_test, he, ga, enhanced],
                          ['Reference', 'Low-light', 'HE', 'Gamma 1/3', 'LLNet']):
    ax.imshow(im, cmap='gray', vmin=0, vmax=1)
    if title != 'Reference':
        p = psnr_fn(img, im, data_range=1.0)
        s = ssim_fn(img, im, data_range=1.0)
        ax.set_title(f'{title}\nPSNR {p:.1f} / SSIM {s:.2f}')
    else:
        ax.set_title('Reference')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Part 6: Bottleneck-feature visualisation / 가중치 시각화

첫 layer 가중치를 17×17 reshape — 논문 Figure 7과 유사하게 blob-like 또는 noise-like 특징이 보일지 확인.

In [ ]:
W0 = model.encoder[0].weight.detach().cpu().numpy()
n_show = 16
idx = np.argsort(np.linalg.norm(W0, axis=1))[-n_show:]
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for k, ax in enumerate(axes.flat):
    if k < n_show:
        f = W0[idx[k]].reshape(PATCH, PATCH)
        f = (f - f.min()) / (f.max() - f.min() + 1e-9)
        ax.imshow(f, cmap='gray')
    ax.axis('off')
plt.suptitle('First-layer LLNet feature detectors (top-norm 16)')
plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | This paper / 이 논문 | Modern equivalent / 현대 등가 |
|---|---|---|
| Synthetic low-light data | random gamma + Gaussian noise on bright images | LOL paired dataset (Wei+ 2018), MIT-Adobe FiveK |
| Architecture | 3-layer SSDA, 17×17 patches | UNet, ResNet, transformer-based LLE |
| Loss | MSE + KL sparsity + L2 | MSE + perceptual + adversarial + Retinex |
| Inference | overlapping patches averaged | full-image FCN inference |
| Paired data needed? | yes (synthetic) | Zero-DCE (no), EnlightenGAN (unpaired) |

**Key takeaway**: 작은 SSDA + 합성 darkening + KL sparsity만으로도 단순 HE / gamma adjustment 보다 잡음 영상에서 더 좋은 결과를 얻을 수 있다. CPU에서 200 iteration으로도 의미 있는 enhancement가 학습됨을 확인.

Even a small SSDA + synthetic darkening + KL sparsity beats simple HE / gamma correction on noisy low-light images, and learns meaningful enhancement in 200 CPU iterations.